In [47]:
"""Data preparation for FinBERT fine-tuning on Financial PhraseBank.
Work done in the notebook
  1. Loads the dataset (sentences_50agree by default)
  2. Cleans it: strips whitespace, removes duplicate sentences and any
     sentence that appears with conflicting labels
  3. Makes a stratified 80/10/10 train/validation/test split (fixed seed)
  4. Saves the splits to data/processed/ so every later script uses the SAME split
 
Cleaning happens BEFORE splitting so the same sentence can never appear in
both train and test (data leakage).
"""

'Data preparation for FinBERT fine-tuning on Financial PhraseBank.\nWork done in the notebook\n  1. Loads the dataset (sentences_50agree by default)\n  2. Cleans it: strips whitespace, removes duplicate sentences and any\n     sentence that appears with conflicting labels\n  3. Makes a stratified 80/10/10 train/validation/test split (fixed seed)\n  4. Saves the splits to data/processed/ so every later script uses the SAME split\n\nCleaning happens BEFORE splitting so the same sentence can never appear in\nboth train and test (data leakage).\n'

In [ ]:
'''Financial Sentiment Classification using FinBERT
Step 2 — Data Preparation

Goal: Clean the raw Financial PhraseBank data and produce a fixed,
leak-free train/validation/test split that every later step (baseline,
FinBERT fine-tuning, evaluation) will reuse.

Input: `FinancialPhraseBank-v1.0/Sentences_50Agree.txt`
(`sentence@label` format, `latin-1` encoded).

Cleaning (done before splitting, to avoid data leakage):
- Removed sentences with conflicting labels (same sentence, different label
  across annotator subsets): 4846 → 4842 rows
- Removed exact duplicate sentences: 4842 → 4836 rows

Split: Stratified 80/10/10 (train/val/test), fixed seed = 42, so class
proportions (~59.4% neutral / 28.1% positive / 12.5% negative) are preserved
across all three splits. Sizes: 3868 / 484 / 484.
Leakage check confirmed: no sentence appears in more than one split.

Tokenization setting decided here (used in Step 4): `max_length = 128`
(standard BERT default) — well above the 99th percentile token length (68)
found in EDA, so truncation affects almost no sentences.

Output: `data/processed/train.csv`, `val.csv`, `test.csv` — the single
source of truth for all later steps.

'''

In [48]:
from pathlib import Path
import pandas as pd
from datasets import Dataset, load_dataset
from sklearn.model_selection import train_test_split

### 1. Loading the Data

In [49]:
CONFIG = "sentences_50agree"
LABELS = {0: "negative", 1: "neutral", 2: "positive"}  # from the dataset card
SEED = 42
PROCESSED_DIR = Path("data/processed")

In [50]:
def load_raw(path: str = "../FinancialPhraseBank-v1.0/Sentences_50Agree.txt") -> pd.DataFrame:
    """Load the raw PhraseBank .txt file (sentence@label format, latin-1 encoded)."""
    rows = []
    with open(path, "r", encoding="latin-1") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            sentence, label_name = line.rsplit("@", 1)
            rows.append((sentence, label_name))

    df = pd.DataFrame(rows, columns=["sentence", "label_name"])
    name_to_label = {"negative": 0, "neutral": 1, "positive": 2}
    df["label"] = df["label_name"].map(name_to_label)
    return df

In [51]:
import os
print(os.getcwd())
print(os.listdir())

c:\Users\utkar\OneDrive\Documents\Desktop\FinBert Project\notebooks
['01_data_explorations.ipynb', '02_data_preprocessing.ipynb', 'results']


### 2. Preprocessing 

In [52]:
# Cleaning 
def clean(df: pd.DataFrame) -> pd.DataFrame:
    """Strip whitespace, drop conflicting-label sentences, drop exact duplicates."""
    df = df.copy()
    df["sentence"] = df["sentence"].str.strip()
    n_start = len(df)
 
    n_labels = df.groupby("sentence")["label"].transform("nunique")
    n_conflicting = int((n_labels > 1).sum())
    df = df[n_labels == 1]  # a sentence with 2 different labels is label noise
 
    df = df.drop_duplicates(subset="sentence", keep="first")
 
    print(f"Rows before cleaning:                {n_start}")
    print(f"Rows removed (conflicting labels):   {n_conflicting}")
    print(f"Rows after removing duplicates:      {len(df)}")
    return df.reset_index(drop=True)

In [53]:
# Make Splits
def make_splits(df: pd.DataFrame, val_size: float = 0.1, test_size: float = 0.1,
                seed: int = SEED):
    """Stratified train/val/test split (stratify = keep class proportions equal)."""
    train_val, test = train_test_split(
        df, test_size=test_size, stratify=df["label"], random_state=seed
    )
    # val_size is a fraction of the WHOLE dataset, so rescale for the second split
    val_fraction = val_size / (1 - test_size)
    train, val = train_test_split(
        train_val, test_size=val_fraction, stratify=train_val["label"], random_state=seed
    )
    return (train.reset_index(drop=True),
            val.reset_index(drop=True),
            test.reset_index(drop=True))

In [54]:
# Save Split
def save_splits(train, val, test, out_dir: Path = PROCESSED_DIR) -> None:
    out_dir.mkdir(parents=True, exist_ok=True)
    for name, part in (("train", train), ("val", val), ("test", test)):
        part.to_csv(out_dir / f"{name}.csv", index=False)

In [55]:
# Load Split
def load_splits(in_dir: Path = PROCESSED_DIR):
    return tuple(pd.read_csv(in_dir / f"{n}.csv") for n in ("train", "val", "test"))

In [56]:
def tokenize_dataframe(df: pd.DataFrame, tokenizer, max_length: int) -> Dataset:
    """Turn a dataframe into a tokenized HF Dataset (input_ids + attention_mask + label).
 
    Truncation is applied here; padding is NOT. Padding is done per batch later
    by DataCollatorWithPadding, which is faster than padding everything to max_length.
    """
    ds = Dataset.from_pandas(df[["sentence", "label"]], preserve_index=False)
    return ds.map(
        lambda batch: tokenizer(batch["sentence"], truncation=True, max_length=max_length),
        batched=True,
        remove_columns=["sentence"],
    )
 

In [57]:
if __name__ == "__main__":
    raw = load_raw()
    cleaned = clean(raw)
    train, val, test = make_splits(cleaned)
    save_splits(train, val, test)
 
    print("\nSplit sizes:", len(train), len(val), len(test))
    for name, part in (("train", train), ("val", val), ("test", test)):
        dist = part["label"].map(LABELS).value_counts(normalize=True).round(3).to_dict()
        print(f"{name:>5} class proportions: {dist}")
 
    # Sanity check: no sentence shared between splits
    assert not set(train["sentence"]) & set(test["sentence"])
    assert not set(train["sentence"]) & set(val["sentence"])
    assert not set(val["sentence"]) & set(test["sentence"])
    print("\nLeakage check passed: no sentence appears in more than one split.")

Rows before cleaning:                4846
Rows removed (conflicting labels):   4
Rows after removing duplicates:      4836

Split sizes: 3868 484 484
train class proportions: {'neutral': 0.594, 'positive': 0.282, 'negative': 0.125}
  val class proportions: {'neutral': 0.593, 'positive': 0.281, 'negative': 0.126}
 test class proportions: {'neutral': 0.593, 'positive': 0.281, 'negative': 0.126}

Leakage check passed: no sentence appears in more than one split.
